# 17 · Dependency-aware structured channel pruning

**Goal:** compare deterministic channel rankings, physically rebuild three multiples-of-eight channel schedules, recover the validation winner with EMA-range QAT, and measure the exact full-INT8 candidate.

## Why dependency propagation is mandatory

A removed pointwise output channel must also remove its BatchNorm entry, the matching channel of the next depthwise kernel, the corresponding input plane of the next pointwise kernel, and—at the final stage—the classifier row. Merely writing zeros leaves dense ESP-NN tensor shapes and latency unchanged.

`rank output channel → slice pointwise output → slice BatchNorm → slice next depthwise channel → slice next pointwise input → repeat`

In [ ]:
import hashlib, json, subprocess, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display

from vww_esp32.config import load_config, seed_everything
from vww_esp32.evaluation import classification_metrics, collect_predictions, expected_calibration_error, select_threshold
from vww_esp32.exporting import convert_full_integer, inspect_tflite, run_tflite
from vww_esp32.modeling import build_tiny_mobilenet_v1, compile_model
from vww_esp32.preprocessing import make_dataset, representative_dataset, split_manifest
from vww_esp32.profiling import profile_model
from vww_esp32.pruning import NAMED_CHANNEL_SCHEDULES, STAGES, select_channel_indices, transfer_structured_weights
from vww_esp32.qat import build_qat_mobilenet_v1, copy_compatible_weights, copy_qat_weights_to_float

config, ROOT = load_config('configs/pruning.yaml')
seed = int(config['project']['seed']); seed_everything(seed)
IMAGE_SIZE = tuple(config['preprocessing']['image_size']); BATCH_SIZE = int(config['preprocessing']['batch_size'])
OUT = ROOT / 'artifacts/pruning/channel'
for child in ('models', 'reports', 'figures', 'logs'): (OUT / child).mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')

## 1 · Load the frozen dataset and Fast-80 teacher

The teacher supplies both initialization tensors and deterministic importance signals. Dataset splits and the held-out-test policy remain unchanged.

In [ ]:
splits = split_manifest(pd.read_csv(ROOT / 'data/processed/manifest.csv'))
assert {key: len(value) for key, value in splits.items()} == {'train': 12000, 'val': 2000, 'test': 2000}
train_ds = make_dataset(splits['train'], IMAGE_SIZE, BATCH_SIZE, training=True, seed=seed)
val_ds = make_dataset(splits['val'], IMAGE_SIZE, BATCH_SIZE, training=False)
test_ds = make_dataset(splits['test'], IMAGE_SIZE, BATCH_SIZE, training=False)
reference = tf.keras.models.load_model(ROOT / config['reference']['float_model'])
reference_metrics = json.loads((ROOT / config['reference']['int8_metrics']).read_text())
_, _, reference_profile = profile_model(reference, training_batch_size=BATCH_SIZE)
print(f"Reference: {reference.count_params():,} params · {reference_profile['estimated_macs_batch1']:,} MACs")

## 2 · Compare channel-importance rules before training

We audit BatchNorm-gamma, pointwise-L1, and their normalized hybrid. Pairwise overlap exposes stages where rankings disagree and where a later sensitivity experiment deserves attention.

In [ ]:
audit_schedule = NAMED_CHANNEL_SCHEDULES['aggressive']
rankings = {method: select_channel_indices(reference, audit_schedule, method) for method in config['channel_pruning']['ranking_methods']}
overlap_rows = []
for stage in STAGES:
    gamma, l1, hybrid = (set(rankings[key][stage]) for key in ('bn_gamma', 'pointwise_l1', 'hybrid'))
    target = len(gamma)
    overlap_rows.append({'stage': stage, 'kept': target, 'gamma_vs_l1': len(gamma & l1) / target, 'gamma_vs_hybrid': len(gamma & hybrid) / target, 'l1_vs_hybrid': len(l1 & hybrid) / target})
overlap = pd.DataFrame(overlap_rows)
overlap.to_csv(OUT / 'reports/ranking_overlap.csv', index=False)
fig, ax = plt.subplots(figsize=(14, 5)); overlap.set_index('stage').drop(columns='kept').plot(kind='bar', ax=ax, color=['#2F80ED', '#21C7E8', '#F2994A']); ax.set(title='Channel-ranking agreement by stage', ylabel='Jaccard-style retained overlap', ylim=(0, 1.05)); fig.tight_layout(); fig.savefig(OUT / 'figures/ranking_overlap.png', dpi=170, bbox_inches='tight'); plt.show()
display(overlap)

## 3 · Build physical channel schedules and verify savings

The first screen fixes the inexpensive, auditable BatchNorm-gamma ranking while varying topology. Counts remain multiples of eight for ESP-NN alignment. Every tensor axis is checked by the transfer utility.

In [ ]:
ranking_method = config['channel_pruning']['selected_initial_ranking']
schedules = {name: tuple(values) for name, values in config['channel_pruning']['schedules'].items()}
candidates, structural_rows = {}, []
for name, schedule in schedules.items():
    model = build_tiny_mobilenet_v1(input_shape=(*IMAGE_SIZE, 3), dropout=config['model']['dropout'], l2=config['model']['l2'], channel_schedule=schedule)
    transfer = transfer_structured_weights(reference, model, schedule, method=ranking_method)
    layers, liveness, profile = profile_model(model, training_batch_size=BATCH_SIZE)
    candidates[name] = model
    structural_rows.append({'candidate': name, 'schedule': '-'.join(map(str, schedule)), 'parameters': profile['parameters'], 'macs': profile['estimated_macs_batch1'], 'mac_reduction_pct': 100 * (1 - profile['estimated_macs_batch1'] / reference_profile['estimated_macs_batch1']), 'peak_live_int8': profile['peak_live_activation_int8_bytes_batch1_hypothetical']})
    (OUT / 'reports' / f'{name}_transfer.json').write_text(json.dumps(transfer, indent=2) + '\n')
structure = pd.DataFrame(structural_rows); structure.to_csv(OUT / 'reports/structural_screen.csv', index=False)
display(structure)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5)); sns.barplot(data=structure, x='candidate', y='mac_reduction_pct', ax=axes[0], color='#21C7E8'); sns.barplot(data=structure, x='candidate', y='peak_live_int8', ax=axes[1], color='#27AE60'); axes[0].set_title('Analytical MAC reduction'); axes[1].set_title('Graph live-activation estimate'); fig.tight_layout(); fig.savefig(OUT / 'figures/channel_structure.png', dpi=170, bbox_inches='tight'); plt.show()

## 4 · Equal-budget sensitivity screen

Each schedule receives the same three-epoch float recovery. The Pareto selector favors validation PR-AUC and F1 while retaining explicit MAC and activation costs; test labels remain untouched.

In [ ]:
screen_rows = []
for name, model in candidates.items():
    compile_model(model, learning_rate=float(config['training']['learning_rate']), label_smoothing=0.0)
    started = time.monotonic(); history = model.fit(train_ds, validation_data=val_ds, epochs=int(config['training']['screening_epochs']), verbose=1)
    y_val, p_val = collect_predictions(model, val_ds)
    threshold = select_threshold(y_val, p_val, objective=config['evaluation']['threshold_objective'], minimum_recall=float(config['evaluation']['minimum_recall']))['threshold']
    metrics = classification_metrics(y_val, p_val, threshold)
    profile_row = structure.set_index('candidate').loc[name]
    metrics.update({'candidate': name, 'macs': int(profile_row.macs), 'peak_live_int8': int(profile_row.peak_live_int8), 'seconds': time.monotonic() - started})
    screen_rows.append(metrics); model.save(OUT / 'models' / f'{name}_screened.keras')
screen = pd.DataFrame(screen_rows).sort_values(['pr_auc', 'f1', 'macs'], ascending=[False, False, True])
screen.to_csv(OUT / 'reports/validation_screen.csv', index=False)
eligible = screen[(screen.recall >= config['evaluation']['minimum_recall']) & (screen.f1 >= config['deployment_gate']['minimum_test_f1'])]
winner_name = (eligible if len(eligible) else screen).iloc[0].candidate
winner_schedule = schedules[winner_name]; winner_float = candidates[winner_name]
display(screen[['candidate', 'macs', 'peak_live_int8', 'accuracy', 'precision', 'recall', 'specificity', 'f1', 'pr_auc']]); print('Selected:', winner_name)

## 5 · EMA-QAT recovery and clean topology restoration

The selected topology—not the original width—is rebuilt with EMA fake quantizers. Learned tensors are copied back into an ordinary dense Keras graph before conversion so deployment contains only standard ESP-NN/TFLM operators.

In [ ]:
qat_model = build_qat_mobilenet_v1(input_shape=(*IMAGE_SIZE, 3), dropout=config['model']['dropout'], l2=config['model']['l2'], range_mode=config['qat']['range_mode'], ema_decay=float(config['qat']['ema_decay']), channel_schedule=winner_schedule)
copy_compatible_weights(winner_float, qat_model)
for layer in qat_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization): layer.trainable = False
for images, _ in train_ds.take(64): _ = qat_model(images, training=True)
compile_model(qat_model, learning_rate=float(config['training']['qat_learning_rate']), label_smoothing=0.0)
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max', patience=int(config['training']['early_stopping_patience']), restore_best_weights=True), tf.keras.callbacks.ReduceLROnPlateau(monitor='val_pr_auc', mode='max', factor=0.3, patience=int(config['training']['reduce_lr_patience']), min_lr=1e-7)]
history = qat_model.fit(train_ds, validation_data=val_ds, epochs=int(config['training']['recovery_epochs']), callbacks=callbacks, verbose=1)
deployment_model = build_tiny_mobilenet_v1(input_shape=(*IMAGE_SIZE, 3), dropout=config['model']['dropout'], l2=config['model']['l2'], channel_schedule=winner_schedule)
strip = copy_qat_weights_to_float(qat_model, deployment_model); assert not strip['skipped_layers'], strip
deployment_model.save(OUT / 'models/channel_pruned_qat_float.keras'); pd.DataFrame(history.history).to_csv(OUT / 'logs/qat_history.csv', index=False)

## 6 · Locked test, full-INT8 export, resource audit, and gates

In [ ]:
y_val, p_val = collect_predictions(deployment_model, val_ds)
threshold = select_threshold(y_val, p_val, objective=config['evaluation']['threshold_objective'], minimum_recall=float(config['evaluation']['minimum_recall']))['threshold']
y_test, p_float = collect_predictions(deployment_model, test_ds); float_metrics = classification_metrics(y_test, p_float, threshold)
model_path = OUT / 'models/vww_channel_pruned_qat_int8.tflite'
convert_full_integer(deployment_model, representative_dataset(splits['train'], IMAGE_SIZE, int(config['export']['representative_samples']), seed), model_path)
contract = inspect_tflite(model_path); test_images = np.concatenate([np.asarray(images) for images, _ in test_ds]); p_int8 = run_tflite(model_path, test_images)
int8_metrics = classification_metrics(y_test, p_int8, threshold); int8_metrics['expected_calibration_error_10_bins'] = expected_calibration_error(y_test, p_int8); int8_metrics['probability_mae_vs_float'] = float(np.mean(np.abs(p_int8 - p_float)))
layers, liveness, model_profile = profile_model(deployment_model, training_batch_size=BATCH_SIZE)
gates = {'full_int8': contract['input']['dtype'] == contract['output']['dtype'] == 'int8', 'mac_target': model_profile['estimated_macs_batch1'] <= config['deployment_gate']['max_macs'], 'recall': int8_metrics['recall'] >= config['deployment_gate']['minimum_test_recall'], 'f1': int8_metrics['f1'] >= config['deployment_gate']['minimum_test_f1'], 'model_bytes': model_path.stat().st_size <= config['deployment_gate']['max_model_bytes']}
summary = {'experiment': 'dependency_aware_channel_pruning', 'selected_candidate': winner_name, 'ranking': ranking_method, 'channel_schedule': list(winner_schedule), 'threshold': threshold, 'float_test_metrics': float_metrics, 'int8_test_metrics': int8_metrics, 'model_profile': model_profile, 'export': contract, 'gates': gates, 'ready_for_device_profile': all(gates.values()), 'model_sha256': hashlib.sha256(model_path.read_bytes()).hexdigest()}
(OUT / 'reports/experiment_summary.json').write_text(json.dumps(summary, indent=2) + '\n'); layers.to_csv(OUT / 'reports/layer_profile.csv', index=False); liveness.to_csv(OUT / 'reports/activation_liveness.csv', index=False)
display(pd.DataFrame([reference_metrics, int8_metrics], index=['Fast-80 PTQ', 'channel-pruned QAT'])[['accuracy', 'precision', 'recall', 'specificity', 'f1', 'pr_auc']]); display(pd.DataFrame({'gate': gates.keys(), 'passed': gates.values()}))

## 7 · Physical ESP32-CAM handoff

The candidate is successful only if measured Invoke latency, tensor-arena use, and complete pipeline FPS improve.

In [ ]:
SERIAL_PORT = '/dev/cu.YOUR_SERIAL_PORT'
command = [str(ROOT / 'scripts/profile_esp32_model.sh'), str(model_path), '80', 'prune_channel_selected', SERIAL_PORT]
print(' '.join(command)); RUN_DEVICE_PROFILE = False
if RUN_DEVICE_PROFILE:
    assert summary['ready_for_device_profile'] and 'YOUR_SERIAL_PORT' not in SERIAL_PORT
    subprocess.run(command, cwd=ROOT, check=True)